# Module 16: Spatial Data Warehousing Lab

In this lab, we explore the "Big Iron" of the modern GIS stack: **Snowflake** and **BigQuery**. 

### **Setup Instructions**
1.  **Credentials**: Copy `.env.example` to a new file named `.env` in the same directory.
2.  **Snowflake**: Fill in your account, user, password, and the specific `DATABASE` and `SCHEMA` you wish to connect to.
3.  **BigQuery**: Provide the path to your Google Cloud Service Account JSON key.

We will demonstrate three patterns:
1. **Zero-Copy Simulation**: Using DuckDB to mock how Data Warehouses query cloud storage directly.
2. **Snowflake Geometry**: Connecting Python to Snowflake and querying `GEOGRAPHY` types.
3. **BigQuery Public Data**: Querying massive public geospatial datasets directly from your workspace.

In [ ]:
import os
import duckdb
import geopandas as gpd
import pandas as pd
import lonboard
import leafmap
from dotenv import load_dotenv
from shapely.geometry import shape

# Load credentials from .env
load_dotenv()

# Pre-checks for Warehouse Connectors
try:
    import snowflake.connector
    HAS_SNOWFLAKE = True
except ImportError:
    HAS_SNOWFLAKE = False
    
try:
    from google.cloud import bigquery
    HAS_BIGQUERY = True
except ImportError:
    HAS_BIGQUERY = False

os.makedirs("data", exist_ok=True)
print(f"Snowflake Connector: {'Installed' if HAS_SNOWFLAKE else 'Not Found'}")
print(f"BigQuery Connector: {'Installed' if HAS_BIGQUERY else 'Not Found'}")

## Phase 1: Zero-Copy Simulation (DuckDB as Mock Warehouse)

Modern warehouses like Snowflake don't require you to "import" data. They use **External Tables** to query GeoParquet files directly on S3. 

We will use DuckDB to simulate this by querying a local Parquet file as if it were an external object store.

In [ ]:
# Direct generation of internal simulation data for stability
local_parquet = "data/simulation_data.parquet"

if not os.path.exists(local_parquet):
    print("Generating simulation data...")
    df_sim = pd.DataFrame({
        'id': [1, 2, 3],
        'name': ['Point A', 'Point B', 'Point C'],
        'longitude': [-122.4194, -122.4313, -122.4018],
        'latitude': [37.7749, 37.7833, 37.7944]
    })
    gdf_sim = gpd.GeoDataFrame(df_sim, geometry=gpd.points_from_xy(df_sim.longitude, df_sim.latitude), crs="EPSG:4326")
    gdf_sim.to_parquet(local_parquet)

# SQL Pattern: Notice we refer directly to the file path, not an internal table
con = duckdb.connect()
query = f"SELECT * FROM read_parquet('{local_parquet}') LIMIT 1000"

df_warehouse = con.execute(query).df()
print(f"Queried {len(df_warehouse)} records directly from 'External Storage'.")
df_warehouse.head()

## Phase 2: Snowflake Integration

If you have a Snowflake account, ensure you have filled in your `.env` file. 
This cell will connect to your specified **Database** and **Schema** and query a spatial table.

In [ ]:
def query_snowflake():
    if not HAS_SNOWFLAKE:
        print("Snowflake connector not found.")
        return None

    user = os.getenv("SNOWFLAKE_USER")
    if not user or user == "your_user":
        print("Skipping Phase 2 (Snowflake): No credentials found in .env")
        return None

    # Connect to Snowflake using .env variables
    ctx = snowflake.connector.connect(
        user=os.getenv("SNOWFLAKE_USER"),
        password=os.getenv("SNOWFLAKE_PASSWORD"),
        account=os.getenv("SNOWFLAKE_ACCOUNT"),
        warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
        database=os.getenv("SNOWFLAKE_DATABASE"),
        schema=os.getenv("SNOWFLAKE_SCHEMA")
    )
    
    # Note: Replace 'my_spatial_table' with your actual table name
    query = "SELECT id, ST_ASWKB(geom) as geometry FROM my_spatial_table LIMIT 100"
    
    df = pd.read_sql(query, ctx)
    df['geometry'] = gpd.GeoSeries.from_wkb(df['geometry'])
    gdf = gpd.GeoDataFrame(df, crs="EPSG:4326")
    
    print(f"Successfully pulled {len(gdf)} records from Snowflake.")
    return gdf

gdf_snowflake = query_snowflake()

## Phase 3: BigQuery & Public Datasets

BigQuery excels at querying multi-petabyte public datasets. If you've pointed `GOOGLE_APPLICATION_CREDENTIALS` to your JSON key, this cell will query the **US Census** data.

In [ ]:
def query_bigquery():
    if not HAS_BIGQUERY:
        print("BigQuery client not found.")
        return None

    cred_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
    if not cred_path or not os.path.exists(cred_path):
        print("Skipping Phase 3 (BigQuery): No service account key found.")
        return None

    from google.cloud import bigquery
    client = bigquery.Client()
    
    # Querying Public US Census Zip Code Boundaries
    query = """
    SELECT 
        zip_code, 
        ST_ASWKB(zip_code_geom) as geometry
    FROM `bigquery-public-data.geo_us_boundaries.zip_codes` 
    WHERE state_code = 'DC'
    LIMIT 100
    """
    
    df = client.query(query).to_dataframe()
    df['geometry'] = gpd.GeoSeries.from_wkb(df['geometry'])
    gdf = gpd.GeoDataFrame(df, crs="EPSG:4326")
    
    print(f"Successfully pulled {len(gdf)} Zip Codes from BigQuery.")
    return gdf

gdf_bigquery = query_bigquery()

## Phase 4: Visualization from Warehouse (Lonboard)

The final step is to merge all active sources into a single interactive map using Lonboard.

In [ ]:
layers = []

# Add Simulation Layer
gdf_sim = gpd.GeoDataFrame(
    df_warehouse, 
    geometry=gpd.points_from_xy(df_warehouse.longitude, df_warehouse.latitude),
    crs="EPSG:4326"
)
layers.extend(lonboard.viz(gdf_sim).layers)

# Add Snowflake Layer if available
if 'gdf_snowflake' in locals() and gdf_snowflake is not None:
    layers.extend(lonboard.viz(gdf_snowflake).layers)

# Add BigQuery Layer if available
if 'gdf_bigquery' in locals() and gdf_bigquery is not None:
    layers.extend(lonboard.viz(gdf_bigquery).layers)

if layers:
    m = lonboard.Map(layers=layers)
    m